# Step 3: Clean (silver layer)

Turn the raw bronze table into analysis-ready views: readable column names, a real date column, volumes in kb/d, and flags for negative or zero months.

In [1]:
from getpass import getpass

import pandas as pd
from sqlalchemy import URL, create_engine, text

# Connect straight to the project database; the password is asked for, never stored
db_url = URL.create(
    "mysql+pymysql",
    username="root",
    password=getpass("MySQL root password: "),
    host="localhost",
    port=3306,
    database="norway_forecast",
)
db = create_engine(db_url)

# Quick check that the bronze table is reachable
with db.connect() as conn:
    row_count = conn.execute(text("SELECT COUNT(*) FROM bronze_field_production_monthly")).scalar()
print(f"Connected. Bronze table has {row_count:,} rows")

Connected. Bronze table has 28,455 rows


In [2]:
# Each field should appear only once per month. If a field-month appeared twice,
# every total we calculate later would double-count it.
duplicate_check = """
SELECT prfNpdidInformationCarrier, prfYear, prfMonth, COUNT(*) AS n_rows
FROM bronze_field_production_monthly
GROUP BY prfNpdidInformationCarrier, prfYear, prfMonth
HAVING COUNT(*) > 1
"""
pd.read_sql(text(duplicate_check), db)

,prfNpdidInformationCarrier,prfYear,prfMonth,n_rows


In [3]:
create_silver_view = """
CREATE OR REPLACE VIEW silver_field_production_monthly AS
SELECT
    prfNpdidInformationCarrier          AS field_id,
    prfInformationCarrier               AS field_name,
    -- Build '2021-8-01' as text, then convert it to a real date (1st of the month)
    STR_TO_DATE(CONCAT(prfYear, '-', prfMonth, '-01'), '%Y-%c-%d') AS production_month,
    prfPrdOilNetMillSm3                 AS oil_mill_sm3,
    prfPrdGasNetBillSm3                 AS gas_bill_sm3,
    prfPrdNGLNetMillSm3                 AS ngl_mill_sm3,
    prfPrdCondensateNetMillSm3          AS condensate_mill_sm3,
    prfPrdOeNetMillSm3                  AS oe_mill_sm3,
    prfPrdProducedWaterInFieldMillSm3   AS water_mill_sm3
FROM bronze_field_production_monthly
"""

with db.begin() as conn:
    conn.execute(text(create_silver_view))
print("View silver_field_production_monthly created")


View silver_field_production_monthly created


In [4]:
# Read a few rows back from the view, newest months first
pd.read_sql(text("""
    SELECT *
    FROM silver_field_production_monthly
    ORDER BY production_month DESC
    LIMIT 10
"""), db)


,field_id,field_name,production_month,oil_mill_sm3,gas_bill_sm3,ngl_mill_sm3,condensate_mill_sm3,oe_mill_sm3,water_mill_sm3
0,23395946,AASTA HANSTEEN,2026-07-01,0.00000,0.53818,0.00000,0.00986,0.54803,0.00173
1,4380167,VOLUND,2026-07-01,0.00405,0.00008,0.00000,0.00000,0.00413,0.04803
2,43807,YME,2026-07-01,0.06162,0.00000,0.00000,0.00000,0.06162,0.13941
3,38542241,ÆRFUGL NORD,2026-07-01,0.00081,0.01288,0.00167,0.00000,0.01536,0.00000
4,42002484,ØRN,2026-07-01,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
5,43765,ÅSGARD,2026-07-01,0.19570,0.41233,0.09230,0.00000,0.70033,0.03608
6,20461008,VISUND SØR,2026-07-01,0.01039,0.04756,0.00682,0.00000,0.06477,0.00000
7,43800,ULA,2026-07-01,0.02179,0.00000,0.00112,0.00000,0.02291,0.25361
8,2834734,URD,2026-07-01,0.00000,0.00000,0.00000,0.00000,0.00000,0.00000
9,3960848,TYRIHANS,2026-07-01,0.03262,0.24159,0.03434,0.00000,0.30855,0.00637


In [5]:
create_rates_view = """
CREATE OR REPLACE VIEW silver_field_production_rates AS
SELECT
    field_id,
    field_name,
    production_month,
    DAY(LAST_DAY(production_month))                                   AS days_in_month,
    -- mill Sm3 per month -> thousand barrels per day: x 1e6 x 6.2898 bbl/Sm3 / days / 1000
    oil_mill_sm3        * 6289.8 / DAY(LAST_DAY(production_month))    AS oil_kbd,
    ngl_mill_sm3        * 6289.8 / DAY(LAST_DAY(production_month))    AS ngl_kbd,
    condensate_mill_sm3 * 6289.8 / DAY(LAST_DAY(production_month))    AS condensate_kbd,
    (oil_mill_sm3 + ngl_mill_sm3 + condensate_mill_sm3)
                        * 6289.8 / DAY(LAST_DAY(production_month))    AS liquids_kbd,
    -- bill Sm3 per month -> mill Sm3 per day
    gas_bill_sm3 * 1000 / DAY(LAST_DAY(production_month))             AS gas_mill_sm3_per_day
FROM silver_field_production_monthly
"""

with db.begin() as conn:
    conn.execute(text(create_rates_view))
print("View silver_field_production_rates created")


View silver_field_production_rates created


In [6]:
# Check 1: one field-month we already worked out by hand (Sverdrup plateau)
print(pd.read_sql(text("""
    SELECT field_name, production_month, days_in_month, ROUND(oil_kbd, 1) AS oil_kbd
    FROM silver_field_production_rates
    WHERE field_name = 'JOHAN SVERDRUP' AND production_month = '2023-12-01'
"""), db))

# Check 2: national total for the latest month (sum over all fields)
pd.read_sql(text("""
    SELECT
        production_month,
        ROUND(SUM(oil_kbd), 1)              AS oil_kbd,
        ROUND(SUM(ngl_kbd), 1)              AS ngl_kbd,
        ROUND(SUM(condensate_kbd), 1)       AS condensate_kbd,
        ROUND(SUM(liquids_kbd), 1)          AS liquids_kbd,
        ROUND(SUM(gas_mill_sm3_per_day), 1) AS gas_mill_sm3_per_day
    FROM silver_field_production_rates
    WHERE production_month = '2026-07-01'
    GROUP BY production_month
"""), db)


       field_name production_month  days_in_month  oil_kbd
0  JOHAN SVERDRUP       2023-12-01             31    747.1


,production_month,oil_kbd,ngl_kbd,condensate_kbd,liquids_kbd,gas_mill_sm3_per_day
0,2026-07-01,1784.1,183.1,19.2,1986.4,351.7


In [7]:
create_rates_view = """
CREATE OR REPLACE VIEW silver_field_production_rates AS
SELECT
    field_id,
    field_name,
    production_month,
    DAY(LAST_DAY(production_month))                                   AS days_in_month,
    -- mill Sm3 per month -> thousand barrels per day: x 1e6 x 6.2898 bbl/Sm3 / days / 1000
    oil_mill_sm3        * 6289.8 / DAY(LAST_DAY(production_month))    AS oil_kbd,
    ngl_mill_sm3        * 6289.8 / DAY(LAST_DAY(production_month))    AS ngl_kbd,
    condensate_mill_sm3 * 6289.8 / DAY(LAST_DAY(production_month))    AS condensate_kbd,
    (oil_mill_sm3 + ngl_mill_sm3 + condensate_mill_sm3)
                        * 6289.8 / DAY(LAST_DAY(production_month))    AS liquids_kbd,
    -- bill Sm3 per month -> mill Sm3 per day
    gas_bill_sm3 * 1000 / DAY(LAST_DAY(production_month))             AS gas_mill_sm3_per_day,
    -- Label unusual months so later steps can decide what to do with them
    CASE
        WHEN oil_mill_sm3 < 0 OR gas_bill_sm3 < 0
          OR ngl_mill_sm3 < 0 OR condensate_mill_sm3 < 0 THEN 'negative'
        WHEN oil_mill_sm3 = 0 AND gas_bill_sm3 = 0
          AND ngl_mill_sm3 = 0 AND condensate_mill_sm3 = 0 THEN 'zero'
        ELSE 'producing'
    END                                                               AS month_status
FROM silver_field_production_monthly
"""

with db.begin() as conn:
    conn.execute(text(create_rates_view))
print("View silver_field_production_rates created")


View silver_field_production_rates created


In [8]:
# How many field-months fall into each status, and across how many fields?
pd.read_sql(text("""
    SELECT
        month_status,
        COUNT(*)                 AS n_months,
        COUNT(DISTINCT field_id) AS n_fields
    FROM silver_field_production_rates
    GROUP BY month_status
    ORDER BY n_months DESC
"""), db)


,month_status,n_months,n_fields
0,producing,26484,132
1,zero,1815,100
2,negative,156,34
